In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

In [2]:
PROJECT_DIR = "/Users/hoyin/Desktop/DanaFarber/workspaces/CMI_Painter_Angiosarcoma_WES_analysis_mh_regional/scripts"
RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DATA_DIR = f"{PROJECT_DIR}/data/processed"


# Preprocessing

## Load medical data and patient reported data and format ID and columns properly

In [3]:
## Load extracted raw data
extracted_medical_data = pd.read_excel(f"{RAW_DATA_DIR}/Batches_1-12_Dates_Notes_Masked (4).xlsx")
patient_reported_data = pd.read_excel(f"{RAW_DATA_DIR}/ASC PRD Cleaned_200 pts.xlsx")

# Formating
extracted_medical_data["STUDY ID"] = extracted_medical_data["STUDY ID"].apply(lambda x: x.replace("ASCPROJECT","ASCProject"))
extracted_medical_data.columns = [col.strip() for col in extracted_medical_data.columns]
extracted_medical_data["PRIMARY SITE"] = extracted_medical_data["PRIMARY SITE"].str.strip()
extracted_medical_data["PRIMARY SITE STANDARDIZED"] = extracted_medical_data["PRIMARY SITE"].replace({
    #"BREAST (PARENCHYMAL)": "BREAST",
    #"BREAST (CUTANEOUS)": "BREAST",
    "TRUNK": "OTHER",
    "NOT FOUND IN RECORD": "NA"
})

extracted_medical_data.loc[
    extracted_medical_data["DX BIOPSY LOCATION"].str.contains("LUNG") & (extracted_medical_data["PRIMARY SITE"] == "VISCERAL ORGAN"),
    "PRIMARY SITE STANDARDIZED"
] = "LUNG"

#print(extracted_medical_data["PRIMARY SITE STANDARDIZED"].value_counts())

extracted_medical_data.loc[
    extracted_medical_data["DX BIOPSY LOCATION"].str.contains("LIVER") & (extracted_medical_data["PRIMARY SITE"] == "VISCERAL ORGAN"),
    "PRIMARY SITE STANDARDIZED"
] = "LIVER"

#print(extracted_medical_data["PRIMARY SITE STANDARDIZED"].value_counts())


extracted_medical_data.loc[
    extracted_medical_data["PRIMARY SITE"].str.contains("BONE"),
    "PRIMARY SITE STANDARDIZED"
] = "BONE"

#print(extracted_medical_data["PRIMARY SITE STANDARDIZED"].value_counts())

extracted_medical_data.loc[
    extracted_medical_data["DX BIOPSY LOCATION"].str.contains("ARM|LEG"),
    "PRIMARY SITE STANDARDIZED"
] = "EXTREMITY"

extracted_medical_data.loc[
    extracted_medical_data["DX BIOPSY LOCATION"].str.contains("CHEST"),
    "PRIMARY SITE STANDARDIZED"
] = "BREAST (CUTANEOUS)"

extracted_medical_data.loc[
    extracted_medical_data["DX BIOPSY LOCATION"].str.contains("BACK"),
    "PRIMARY SITE STANDARDIZED"
] = "OTHER"

extracted_medical_data.loc[
    extracted_medical_data["PRIMARY SITE STANDARDIZED"].str.contains("OTHER"),
    "PRIMARY SITE STANDARDIZED"
] = "OTHER"

extracted_medical_data["RAAS"] = extracted_medical_data["RAAS"].replace({"YES":1,"NO":0,"NOT FOUND IN RECORD":np.NaN})
extracted_medical_data["LAAS"] = extracted_medical_data["LAAS/STEWART-TREVES"].replace({"YES":1,"NO":0,"NOT FOUND IN RECORD":np.NaN})
extracted_medical_data["CUTANEOUS AS"] = extracted_medical_data["CUTANEOUS AS"].replace({"YES":1,"NO":0,"NOT FOUND IN RECORD":np.NaN})
extracted_medical_data["LOCAL RECURRENCE"] = extracted_medical_data["LOCAL RECURRENCE"].replace({"YES":1,"NO":0,"NOT FOUND IN RECORD":np.NaN})
extracted_medical_data["HAS METS AT DX"] = extracted_medical_data["METS SITES AT METS DX"].replace({
    "NOT METS": "NO METS",
})

extracted_medical_data.loc[
    extracted_medical_data["HAS METS AT DX"] != "NO METS","HAS METS AT DX"
] = 1
extracted_medical_data.loc[
    extracted_medical_data["HAS METS AT DX"] == "NO METS","HAS METS AT DX"
] = 0
extracted_medical_data.loc[
    extracted_medical_data["METS SITES AT METS DX"] == "NOT FOUND IN RECORD","HAS METS AT DX"
] = np.NaN

extracted_medical_rename_dict = {
    col:col + " (EHR_EXTRACTED)" for col in extracted_medical_data.columns if col != "STUDY ID"
}
extracted_medical_data = extracted_medical_data.rename(columns=extracted_medical_rename_dict)

def format_asc_id(string):
    return "ASCProject_" + str(string).zfill(4)

patient_reported_data["STUDY ID"] = patient_reported_data["legacy_shortid"].apply(format_asc_id)
patient_reported_data_rename_dict = {
    col:col + " (PRD)" for col in patient_reported_data.columns if col != "STUDY ID"
}
patient_reported_data = patient_reported_data.rename(columns=patient_reported_data_rename_dict)
patient_reported_data


,legacy_shortid (PRD),PREQUAL_SELF_DESCRIBE (PRD),AGE OF DIAGNOSIS (PRD),DIAGNOSIS_LOC (PRD),DIAGNOSIS_LOC_BONELIMB_DETAILS (PRD),DIAGNOSIS_LOC_ABDOMINAL_DETAILS (PRD),DIAGNOSIS_LOC_OTHER_DETAILS (PRD),EVER_LOCATION (PRD),EVER_LOCATION_BONELIMB_DETAILS (PRD),EVER_LOCATION_ABDOMINAL_DETAILS (PRD),EVER_LOCATION_OTHER_DETAILS (PRD),CURRENT_LOCATION (PRD),CURRENT_LOCATION_BONELIMB_DETAILS (PRD),CURRENT_LOCATION_ABDOMINAL_DETAILS (PRD),CURRENT_LOCATION_OTHER_DETAILS (PRD),SURGERY (PRD),SURGERY_CLEAN_MARGINS (PRD),RADIATION (PRD),RADIATION_SURGERY (PRD),ALL_TREATMENTS (PRD),CURRENTLY_TREATED (PRD),CURRENT_THERAPIES (PRD),OTHER_CANCER (PRD),OTHER_CANCER_LIST (PRD),OTHER_CANCER_RADIATION (PRD),OTHER_CANCER_RADIATION_LOC (PRD),REFERRAL_SOURCE (PRD),EXPERIENCE (PRD),HISPANIC (PRD),RACE (PRD),RACE_OTHER_DETAILS (PRD),BIRTH_YEAR (PRD),COUNTRY (PRD),POSTAL_CODE (PRD),DIAGNOSIS_PRIMARY_LOC (PRD),DIAGNOSIS_PRIMARY_LOC_OTHER_DETAILS (PRD),DIAGNOSIS_SPREAD (PRD),DIAGNOSIS_SPREAD_LOC (PRD),DIAGNOSIS_SPREAD_LOC_BONELIMB_DETAILS (PRD),POST_DIAGNOSIS_SPREAD (PRD),POST_DIAGNOSIS_SPREAD_LOC (PRD),POST_DIAGNOSIS_SPREAD_LOC_BONELIMB_DETAILS (PRD),POST_DIAGNOSIS_SPREAD_LOC_OTHER_DETAILS (PRD),LOCAL_RECURRENCE (PRD),TREATMENT_PAST (PRD),TREATMENT_NOW (PRD),DISEASE_FREE_NOW (PRD),SUPPORT_MEMBERSHIP (PRD),SUPPORT_MEMBERSHIP_TEXT (PRD),CONSENT_DOB (PRD),STUDY ID
0,3,DIAGNOSED,38.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YES,YES,YES,AFTER,NaN,NaN,NaN,YES,mCRC;2014,YES,"Pelvis, ankle, c1/c2 neck",NaN,FUAS,NaN,NaN,NaN,NaN,NaN,NaN,BREAST,NaN,NO,NaN,NaN,NO,NaN,NaN,NaN,NO,"AIM, Gemzar/Taxotere, Gemzar alone, Taxol",NaN,YES,YES,NaN,1973-03-24,ASCProject_0003
1,5,DIAGNOSED,31.073973,BREAST,NaN,NaN,NaN,"BREAST,OTHER",NaN,NaN,Lower right flank,"BREAST,OTHER",NaN,NaN,Right posterior shoulder,YES,YES,YES,BEFORE,"Gem/tax, AIM, taxol,",YES,tremelimumab/durvalumab clinical trial immunot...,NO,NaN,NO,NaN,Facebook,glad to have lived long enough to help others ...,NO,WHITE,NaN,NaN,NaN,NaN,BREAST,NaN,NO,NaN,NaN,YES,"BREAST,OTHER",NaN,lower right flank (back) nad toher breast (right),NO,"chemo, radiation, and surgery",clinical trial -dualimmunotherapy,NO,YES,FB...local sarcoma warriors get together yearl...,1974-06-12,ASCProject_0005
2,6,DIAGNOSED,29.882192,"HEART,LUNG",NaN,NaN,NaN,"HEART,LUNG",NaN,NaN,NaN,LUNG,NaN,NaN,NaN,NO,NaN,NO,NaN,"taxol, gemcitabine + docetaxol, doxorubicin, o...",YES,olaratumumab + doxirubicin,NO,NaN,NO,NaN,Corrie Painter,NaN,NO,WHITE,NaN,NaN,NaN,NaN,HEART,NaN,YES,LUNG,NaN,DK,NaN,NaN,NaN,DK,NaN,"taxol, Gem taxol, and others",NO,NO,NaN,1986-07-22,ASCProject_0006
3,7,DIAGNOSED,41.868493,OTHER,NaN,NaN,Bladder,"LIVER,SPLEEN,LUNG,LYMPH,BONELIMB,ABDOMINAL",Femur / pelvis,"Large intestine , small intestine , sigmoid colon",NaN,"LIVER,SPLEEN,LUNG,LYMPH,BONELIMB,ABDOMINAL",Femur / pelvis,Intestines,NaN,YES,YES,YES,AFTER,"Aim , gem tax , trc 105 , taxol",NO,NaN,YES,thyroid cancer;2015,YES,Thyroid,Internet,I have the kdr mutation i have never gone more...,NO,BLACK,NaN,1973.0,US,19335,OTHER,bladder,NO,NaN,NaN,NO,NaN,NaN,NaN,YES,Chemotherapy treatment aim,Dixotaxle,YES,YES,Nothing,1973-06-28,ASCProject_0007
4,8,DIAGNOSED,39.939726,LIVER,NaN,NaN,NaN,LIVER,NaN,NaN,NaN,NED,NaN,NaN,NaN,YES,YES,NO,NaN,Taxotere and Gemzar,NO,NaN,YES,Bladder Cancer;2004,NO,NaN,Angiosarcoma cancer Facebook group,My AS was found during my regular follow up ch...,NO,SOUTH_EAST_ASIAN,NaN,NaN,NaN,NaN,LIVER,NaN,NO,NaN,NaN,NO,NaN,NaN,NaN,NO,Surgery and Chemo,None,YES,YES,My FB support group for AS is one of the best ...,1974-04-02,ASCProject_0008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,PXPTMJ,DIAGNOSED,79.000000,HEADFACENECK,NaN,NaN,NaN,HEADFACENECK,NaN,NaN,NaN,NED,NaN,NaN,NaN,YES,YES,YES,AFTER,none,YES,awaiting proshetic nose next week,YES,squamas cell about 2007 basel cell numerous t...,NO,NaN,angio sarcoma fac

In [11]:
# extracted_medical_data[
#     extracted_medical_data["PRIMARY SITE STANDARDIZED (EHR_EXTRACTED)"]=="VISCERAL ORGAN"
# ]
# extracted_medical_data[
#     extracted_medical_data["PRIMARY SITE STANDARDIZED (EHR_EXTRACTED)"]=="OTHER"
# ]
# extracted_medical_data[
#     extracted_medical_data["PRIMARY SITE STANDARDIZED (EHR_EXTRACTED)"]=="EXTREMITY"
# ]

,STUDY ID,BATCH NUMBER (EHR_EXTRACTED),AGE AT DX (EHR_EXTRACTED),SEX (EHR_EXTRACTED),PRIMARY SITE (EHR_EXTRACTED),DX BIOPSY LOCATION (EHR_EXTRACTED),RAAS (EHR_EXTRACTED),LAAS/STEWART-TREVES (EHR_EXTRACTED),CUTANEOUS AS (EHR_EXTRACTED),LOCAL RECURRENCE (EHR_EXTRACTED),LOCAL RECURRENCE SITE (EHR_EXTRACTED),LOCAL RECURRENCE DATE 1 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 2 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 3 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 4 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 5 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 6 (DAYS FROM DX) (EHR_EXTRACTED),DATE METS DX (DAYS FROM DX) (EHR_EXTRACTED),METS SITES AT METS DX (EHR_EXTRACTED),METS SITES EVER (EHR_EXTRACTED),FINAL DATE MEDR (DAYS FROM DX) (EHR_EXTRACTED),DATE ABSTRACTED (EHR_EXTRACTED),ABSTRACTED FROM (EHR_EXTRACTED),ABSTRACTOR (EHR_EXTRACTED),PRIMARY SITE STANDARDIZED (EHR_EXTRACTED),LAAS (EHR_EXTRACTED),HAS METS AT DX (EHR_EXTRACTED)
18,ASCProject_0095,3,47,F,OTHER (RIGHT GLUTE),RIGHT GLUTEAL MASS,1.0,NO,1.0,1.0,NOT FOUND IN RECORD,946,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,2011,2021-06-23,"95_1,2,3,4,5,6,7,8,9,10,11,12,13,14",SS,OTHER,0.0,0
31,ASCProject_0357,3,62,M,OTHER (MEDIASTINUM),MEDIASTINUM,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,94,2020-01-14,"357_DFCI_FULL, ASC_357_T1_SK, ASC_357_YALE",QC/AD,OTHER,0.0,0
69,ASCProject_0068,5,55,F,"OTHER (LEFT PELVIS, RETROPERITONEUM)",LEFT PELVIC/RETROPERITONEAL MASS,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,1883,2021-07-16,"68_1, 68_2",SS,OTHER,0.0,0
101,ASCProject_0065,9,60,F,"OTHER (LEFT RETROPERITONEUM, ADRENAL GLAND)",LEFT RETROPERITONEAL MASS/ADRENAL GLAND,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,2501,2022-01-19,"ASC_65_mdanderson_full, ASC_65_pathreport, ASC...",IS,OTHER,0.0,0
139,ASCProject_PRDZ3M,9,51,M,SKIN-CUTANEOUS (ASSOCIATED WITH LYMPHEDEMA),ERYTHEMATOUS LESION ON LEFT SCAPULAR/LEFT UPPE...,0.0,YES,1.0,1.0,EXTREMITY,1333,NaN,NaN,NaN,NaN,NaN,1554,LUNG,"LUNG, DISTANT LYMPH NODE, BONE, TRUNK",1773,2022-02-07,"ASC_PRDZ3M_MSKCC, ASC_PRDZ3M_T2_PATHREPORT",QC,OTHER,1.0,1
147,ASCProject_0034,12,57,M,OTHER [PENIS],SCROTUM,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,1582,2023-03-07,34_UC_Davis,IS,OTHER,0.0,0
151,ASCProject_0168,12,20,M,OTHER [PRIMARY SITE UNKNOWN],LIVER,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,"LIVER, LUNG, DISTANT LYMPH NODE, VISCERAL ORGAN","LIVER, LUNG, DISTANT LYMPH NODE, VISCERAL ORGAN",217,2023-03-15,"168_OCR, 168_Princeton_2, 168_Princeton, ASC_1...",IS,OTHER,0.0,1
154,ASCProject_0214,12,67,M,TRUNK,MID BACK,0.0,NO,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,511,2023-03-08,"ASC_2124_ERLANGER.MEDICAL_NO_RECORD, ASC_214_E...",HG,OTHER,0.0,0
163,ASCProject_0353,12,68,F,OTHER[BRAIN],BRAIN,0.0,NO,0.0,1.0,OTHER[BRAIN],0,NaN,NaN,NaN,NaN,NaN,NaN,NOT METS,NO METS,721,2023-03-25,"ASC_PRJCV3_SpringfieldNeuro_OCR.pdf, ASC_353_B...",OA,OTHER,0.0,0


In [9]:
extracted_medical_data["PRIMARY SITE STANDARDIZED (EHR_EXTRACTED)"].value_counts()

HNFS                    42
BREAST (PARENCHYMAL)    41
BREAST (CUTANEOUS)      39
EXTREMITY               13
CARDIAC                  9
OTHER                    9
VISCERAL ORGAN           5
SPLEEN                   5
LIVER                    4
BONE                     4
LUNG                     3
NA                       2
Name: PRIMARY SITE STANDARDIZED (EHR_EXTRACTED), dtype: int64

## Use MED EXTRACTED whenever possible and PRD if not

In [58]:
merged_clin_data = extracted_medical_data.merge(patient_reported_data,on="STUDY ID",how="outer")
merged_clin_data["EHR_EXTRACTED"] = merged_clin_data["STUDY ID"].isin(extracted_medical_data["STUDY ID"])

## Combine Age
merged_clin_data["Age (Combined)"] = merged_clin_data["AGE AT DX (EHR_EXTRACTED)"].fillna(
    merged_clin_data["AGE OF DIAGNOSIS (PRD)"]
).replace("NOT FOUND IN RECORD",np.NaN).astype(float)

## Combine Primary Sites
merged_clin_data["PRIMARY SITE (Combined)"] = merged_clin_data["PRIMARY SITE STANDARDIZED (EHR_EXTRACTED)"].fillna(
    merged_clin_data["DIAGNOSIS_LOC (PRD)"]
).replace("NOT FOUND IN RECORD",np.NaN)

multiple_primaries_combined_index = merged_clin_data["PRIMARY SITE (Combined)"].apply(lambda x: len(x.split(",")) > 1)

merged_clin_data.loc[
    multiple_primaries_combined_index,
    "PRIMARY SITE (Combined)"
] = np.NaN

merged_clin_data.loc[
    merged_clin_data["PRIMARY SITE (Combined)"] == "NA",
    "PRIMARY SITE (Combined)"
] = np.NaN

merged_clin_data.loc[
    (merged_clin_data["PRIMARY SITE (Combined)"] == "BREAST"),
    "PRIMARY SITE (Combined)"
] = "BREAST (NOS)"

merged_clin_data.loc[
    (merged_clin_data["PRIMARY SITE (Combined)"] == "ABDOMINAL"),
    "PRIMARY SITE (Combined)"
] = "VISCERAL ORGAN"

merged_clin_data.loc[
    (merged_clin_data["PRIMARY SITE (Combined)"] == "BONE") | (merged_clin_data["PRIMARY SITE (Combined)"] == "EXTREMITY"),
    "PRIMARY SITE (Combined)"
] = "BONELIMB"

merged_clin_data.loc[
    (merged_clin_data["PRIMARY SITE (Combined)"] == "HEADFACENECK") | (merged_clin_data["PRIMARY SITE (Combined)"] == "SCALP"),
    "PRIMARY SITE (Combined)"
] = "HNFS"

merged_clin_data.loc[
    (merged_clin_data["PRIMARY SITE (Combined)"] == "CARDIAC"),
    "PRIMARY SITE (Combined)"
] = "HEART"


merged_clin_data["PRIMARY SITE (Combined)"].value_counts()


HNFS                    60
BREAST (PARENCHYMAL)    41
BREAST (CUTANEOUS)      39
BREAST (NOS)            33
BONELIMB                22
OTHER                   17
HEART                   14
VISCERAL ORGAN           6
SPLEEN                   5
LUNG                     4
LIVER                    4
Name: PRIMARY SITE (Combined), dtype: int64

In [59]:
merged_clin_data["PRIMARY SITE (Combined)"].value_counts().sum()

245

## Make RAAS/LAAS classifications

In [60]:
def make_raas_laas_classification(row):
    if (row["LAAS (EHR_EXTRACTED)"] != row["LAAS (EHR_EXTRACTED)"]) and (row["RAAS (EHR_EXTRACTED)"] != row["RAAS (EHR_EXTRACTED)"]):
        return np.NaN
    laas_flag = row["LAAS (EHR_EXTRACTED)"] == 1
    raas_flag = row["RAAS (EHR_EXTRACTED)"] == 1
    if raas_flag and laas_flag:
        return "RAAS & LAAS"
    elif laas_flag:
        return "LAAS"
    elif raas_flag:
        return "RAAS"
    else:
        return "Non-RAAS/LAAS"

raas_laas_classes = [make_raas_laas_classification(row) for idx,row in merged_clin_data.iterrows()]
merged_clin_data["RAAS_LAAS_Class"] = raas_laas_classes
merged_clin_data["RAAS_LAAS_Class"].value_counts()

Non-RAAS/LAAS    127
RAAS              41
RAAS & LAAS        3
LAAS               3
Name: RAAS_LAAS_Class, dtype: int64

## Count number of metastatisis

In [61]:
def num_met_sites(string):
    if string == "NO METS":
        return 0
    elif string == "NOT FOUND IN RECORD":
        return np.NaN
    elif string == string:
        return len(string.split(","))
    else:
        return np.NaN

merged_clin_data["NUM MET SITES EVER (EHR_EXTRACTED)"] = merged_clin_data["METS SITES EVER (EHR_EXTRACTED)"].apply(num_met_sites)
merged_clin_data["NUM MET SITES EVER (EHR_EXTRACTED)"].value_counts()



0.0    92
1.0    26
2.0    17
4.0    10
3.0     9
5.0     2
7.0     1
6.0     1
Name: NUM MET SITES EVER (EHR_EXTRACTED), dtype: int64

In [64]:
merged_clin_data.to_csv(f"{PROCESSED_DATA_DIR}/clinical_data.tsv",sep="\t",index=False)